In [15]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the dataset
dataset_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single'

# Check if the path exists and list the first few files
if os.path.exists(dataset_path):
    print(f"Successfully accessed: {dataset_path}")
    files = os.listdir(dataset_path)
    print(f"Total items found: {len(files)}")
    print("Sample files:", files[:5])
else:
    print(f"Path not found: {dataset_path}. Please ensure the folder name is correct.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully accessed: /content/drive/MyDrive/Colab Notebooks/MVSA_Single
Total items found: 4
Sample files: ['data', 'labelResultAll.txt', 'mvsa_single_processed.parquet', 'checkpoints']


In [16]:
import pandas as pd
import os
from PIL import Image
import io
import gc
from concurrent.futures import ThreadPoolExecutor

# Paths
labels_file = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/labelResultAll.txt'
data_dir = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/data'
# Path ke file yang baru saja disimpan
parquet_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet'

def process_single_row(row):
    """Helper function to process one image-text pair with resizing."""
    try:
        item_id = str(int(row['id']))
        image_path = os.path.join(data_dir, f"{item_id}.jpg")
        text_path = os.path.join(data_dir, f"{item_id}.txt")

        t_label = str(row['text']).strip()
        i_label = str(row['image']).strip()

        # Sentiment logic
        if t_label == "neutral":
            f_label = i_label
        elif i_label == 'neutral':
            f_label = t_label
        elif t_label == i_label:
            f_label = t_label
        else:
            return None

        if os.path.exists(image_path) and os.path.exists(text_path):
            # Process and Resize Image to 384x384 (standard for ViLT)
            with Image.open(image_path) as img:
                img = img.convert('RGB')
                img = img.resize((384, 384), Image.Resampling.LANCZOS)
                buf = io.BytesIO()
                img.save(buf, format='PNG')
                img_bytes = buf.getvalue()

            # Process Text
            with open(text_path, 'r', encoding='latin-1') as file:
                text_content = file.read().strip()

            return {
                'ID': item_id,
                'text_sentiment': t_label,
                'image_sentiment': i_label,
                'final_sentiment': f_label,
                'image_bytes': img_bytes,
                'text_content': text_content
            }
    except Exception as e:
        print(f"Error processing row: {e}")
        return None
    return None

# Cek apakah file parquet sudah ada
if os.path.exists(parquet_path):
    print(f"File parquet sudah ada, memuat dari: {parquet_path}")
    df = pd.read_parquet(parquet_path)
    print(f"Total data dimuat: {len(df)}")
    print("\nDistribusi sentimen:")
    print(df['final_sentiment'].value_counts())
    display(df.head())
else:
    print(f"File parquet tidak ditemukan, memulai pemrosesan...")
    print(f"Membaca labels dari: {labels_file}")

    # 1. Read labels
    df_labels = pd.read_csv(labels_file, sep=',', header=0)
    print(f"Total labels dibaca: {len(df_labels)}")
    print(f"Nama kolom dalam file labels: {list(df_labels.columns)}")
    print(f"Sample data:\n{df_labels.head()}")

    # 2. Parallel processing
    print("\nMemulai pemrosesan paralel dengan resizing gambar ke 384x384...")
    rows = [row for _, row in df_labels.iterrows()]

    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(process_single_row, rows))

    # 3. Filter results and cleanup
    processed_records = [r for r in results if r is not None]
    print(f"Total records berhasil diproses: {len(processed_records)}")

    if len(processed_records) > 0:
        df = pd.DataFrame(processed_records)

        # 4. Simpan ke parquet
        print(f"Menyimpan data ke: {parquet_path}")
        df.to_parquet(parquet_path, index=False)
        print("Data berhasil disimpan!")

        del results, processed_records, df_labels, rows
        gc.collect()

        print(f"\nTotal data berhasil diproses dan disimpan: {len(df)}")
        print("\nDistribusi sentimen:")
        print(df['final_sentiment'].value_counts())
        display(df.head())
    else:
        print("PERINGATAN: Tidak ada data yang berhasil diproses!")
        print("Periksa apakah:")
        print(f"  1. File label ada di: {labels_file}")
        print(f"  2. Folder data ada di: {data_dir}")
        print(f"  3. File .jpg dan .txt ada di dalam folder data")

File parquet sudah ada, memuat dari: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total data dimuat: 4511

Distribusi sentimen:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."


In [17]:
import pandas as pd

# Tentukan path penyimpanan di Google Drive
parquet_save_path = '/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet'

try:
    # Menyimpan DataFrame ke format Parquet
    # Format ini efisien untuk menyimpan data biner seperti image_bytes
    df.to_parquet(parquet_save_path, index=False)
    print(f"Dataset berhasil disimpan ke: {parquet_save_path}")
    print(f"Ukuran file: {os.path.getsize(parquet_save_path) / (1024*1024):.2f} MB")
except Exception as e:
    print(f"Terjadi kesalahan saat menyimpan: {e}")

Dataset berhasil disimpan ke: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Ukuran file: 853.77 MB


In [18]:
# ─────────────────────────────────────────────
# Cell 3 — Install & Import Dependencies
# ─────────────────────────────────────────────
!pip install transformers accelerate scikit-learn pyarrow torchvision -q

import os, io, gc, random, warnings
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import torchvision.models as tv_models
import torchvision.transforms as transforms

from transformers import BertModel, BertTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings('ignore')
print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


All libraries imported successfully.
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU Memory      : 15.6 GB


In [19]:
# ─────────────────────────────────────────────
# Cell 4 — Model Configuration
# ─────────────────────────────────────────────
CONFIG = {
    # ── Paths ──────────────────────────────────────────
    "parquet_path"      : "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet",
    "checkpoint_dir"    : "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/checkpoints",

    # ── Text Encoder ───────────────────────────────────
    "bert_model_name"   : "bert-base-uncased",
    "bert_hidden_size"  : 768,
    "bert_num_layers"   : 12,
    "bert_num_heads"    : 12,         # BERT internal heads (not co-attn)
    "max_text_length"   : 128,

    # ── Image Encoder ──────────────────────────────────
    "image_backbone"    : "resnet101",
    "image_feature_dim" : 2048,       # ResNet-101 last conv output channels
    "image_proj_dim"    : 768,        # projected to match BERT hidden size
    "image_input_size"  : 224,        # resize for ResNet

    # ── Co-Attention ───────────────────────────────────
    "co_attn_type"      : "multi-head",   # bidirectional cross-attention
    "co_attn_layers"    : 2,
    "co_attn_heads"     : 8,          # multi-head count per co-attn layer
    "co_attn_dropout"   : 0.1,
    "ffn_hidden_dim"    : 3072,       # feed-forward inner dim (4× 768)

    # ── Fusion & Classifier ────────────────────────────
    "fusion_strategy"   : "concatenation",   # concat CLS + mean-pooled image
    "fusion_input_dim"  : 768 * 2,    # 1536
    "classifier_hidden" : 512,
    "num_classes"       : 3,          # negative / neutral / positive
    "classifier_dropout": 0.3,

    # ── Training ───────────────────────────────────────
    "epochs"            : 20,
    "batch_size"        : 16,
    "learning_rate"     : 2e-5,
    "bert_lr"           : 2e-5,
    "new_params_lr"     : 1e-4,       # lr × 5 for non-BERT params
    "weight_decay"      : 1e-2,
    "warmup_ratio"      : 0.1,        # 10 % of total steps
    "gradient_clip"     : 1.0,
    "early_stop_patience": 4,

    # ── Data Split ─────────────────────────────────────
    "train_ratio"       : 0.80,
    "val_ratio"         : 0.10,
    "test_ratio"        : 0.10,

    # ── Reproducibility ────────────────────────────────
    "seed"              : 42,
    "num_workers"       : 2,
    "device"            : "cuda" if torch.cuda.is_available() else "cpu",
}

# ── Seed everything ──────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

# ── Pretty-print configuration ───────────────────────────────────────
SEP = "=" * 60
print(SEP)
print("  ViLBERT Multimodal Sentiment Analysis — Configuration")
print(SEP)
print(f"  Device              : {CONFIG['device'].upper()}")
if torch.cuda.is_available():
    print(f"  GPU                 : {torch.cuda.get_device_name(0)}")

print(f"\n  ── Text Encoder ──────────────────────────────────────")
print(f"  Model               : {CONFIG['bert_model_name']}")
print(f"  Hidden size         : {CONFIG['bert_hidden_size']}")
print(f"  BERT layers         : {CONFIG['bert_num_layers']}")
print(f"  BERT attention heads: {CONFIG['bert_num_heads']}")
print(f"  Max token length    : {CONFIG['max_text_length']}")

print(f"\n  ── Image Encoder ─────────────────────────────────────")
print(f"  Backbone            : {CONFIG['image_backbone']}")
print(f"  Feature dim (raw)   : {CONFIG['image_feature_dim']}")
print(f"  Projected dim       : {CONFIG['image_proj_dim']}")
print(f"  Input image size    : {CONFIG['image_input_size']}×{CONFIG['image_input_size']}")

print(f"\n  ── Co-Attention Mechanism ────────────────────────────")
print(f"  Type                : {CONFIG['co_attn_type']} (bidirectional)")
print(f"  Number of layers    : {CONFIG['co_attn_layers']}")
print(f"  Number of heads     : {CONFIG['co_attn_heads']}")
print(f"  FFN hidden dim      : {CONFIG['ffn_hidden_dim']}")
print(f"  Dropout             : {CONFIG['co_attn_dropout']}")

print(f"\n  ── Fusion & Classifier ───────────────────────────────")
print(f"  Fusion strategy     : {CONFIG['fusion_strategy']}")
print(f"  Fusion input dim    : {CONFIG['fusion_input_dim']}")
print(f"  Classifier hidden   : {CONFIG['classifier_hidden']}")
print(f"  Num output classes  : {CONFIG['num_classes']}  (neg / neu / pos)")
print(f"  Classifier dropout  : {CONFIG['classifier_dropout']}")

print(f"\n  ── Training Hyperparameters ──────────────────────────")
print(f"  Epochs              : {CONFIG['epochs']}")
print(f"  Batch size          : {CONFIG['batch_size']}")
print(f"  BERT LR             : {CONFIG['bert_lr']}")
print(f"  New-params LR       : {CONFIG['new_params_lr']}")
print(f"  Weight decay        : {CONFIG['weight_decay']}")
print(f"  Warmup ratio        : {CONFIG['warmup_ratio']} ({int(CONFIG['warmup_ratio']*100)}% of steps)")
print(f"  Gradient clip       : {CONFIG['gradient_clip']}")
print(f"  Early-stop patience : {CONFIG['early_stop_patience']} epochs")

print(f"\n  ── Data ──────────────────────────────────────────────")
print(f"  Split (train/val/test): {CONFIG['train_ratio']}/{CONFIG['val_ratio']}/{CONFIG['test_ratio']}")
print(f"  Random seed         : {CONFIG['seed']}")
print(SEP)


  ViLBERT Multimodal Sentiment Analysis — Configuration
  Device              : CUDA
  GPU                 : Tesla T4

  ── Text Encoder ──────────────────────────────────────
  Model               : bert-base-uncased
  Hidden size         : 768
  BERT layers         : 12
  BERT attention heads: 12
  Max token length    : 128

  ── Image Encoder ─────────────────────────────────────
  Backbone            : resnet101
  Feature dim (raw)   : 2048
  Projected dim       : 768
  Input image size    : 224×224

  ── Co-Attention Mechanism ────────────────────────────
  Type                : multi-head (bidirectional)
  Number of layers    : 2
  Number of heads     : 8
  FFN hidden dim      : 3072
  Dropout             : 0.1

  ── Fusion & Classifier ───────────────────────────────
  Fusion strategy     : concatenation
  Fusion input dim    : 1536
  Classifier hidden   : 512
  Num output classes  : 3  (neg / neu / pos)
  Classifier dropout  : 0.3

  ── Training Hyperparameters ────────────────

In [20]:
# ─────────────────────────────────────────────
# Cell 5 — Load Parquet & Prepare Data Splits
# ─────────────────────────────────────────────
print(f"Loading parquet from: {CONFIG['parquet_path']}")
df_full = pd.read_parquet(CONFIG["parquet_path"])
print(f"Total rows loaded : {len(df_full)}")
print(f"Columns           : {list(df_full.columns)}")

# ── Drop any NaN rows ────────────────────────────────────────────────
before = len(df_full)
df_full = df_full.dropna(subset=["text_content", "image_bytes", "final_sentiment"]).reset_index(drop=True)
print(f"Rows after dropna : {len(df_full)}  (dropped {before - len(df_full)})")

# ── Encode labels: negative→0, neutral→1, positive→2 ────────────────
LABEL_MAP  = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_IMAP = {v: k for k, v in LABEL_MAP.items()}

df_full["label"] = df_full["final_sentiment"].str.strip().map(LABEL_MAP)
missing_labels = df_full["label"].isna().sum()
if missing_labels > 0:
    print(f"WARN: {missing_labels} rows have unrecognised labels — dropping them.")
    df_full = df_full.dropna(subset=["label"]).reset_index(drop=True)
df_full["label"] = df_full["label"].astype(int)

print(f"\nLabel distribution (final_sentiment):")
for lbl, idx in LABEL_MAP.items():
    count = (df_full["label"] == idx).sum()
    pct   = count / len(df_full) * 100
    print(f"  {idx}  {lbl:<10}  {count:>5}  ({pct:.1f}%)")

# ── Stratified split: 80 / 10 / 10 ─────────────────────────────────
df_train, df_tmp = train_test_split(
    df_full,
    test_size=(CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_full["label"],
    random_state=CONFIG["seed"],
)
df_val, df_test = train_test_split(
    df_tmp,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_tmp["label"],
    random_state=CONFIG["seed"],
)

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f"\nData split (stratified):")
print(f"  Train : {len(df_train):>5} rows")
print(f"  Val   : {len(df_val):>5} rows")
print(f"  Test  : {len(df_test):>5} rows")
print(f"  Total : {len(df_train)+len(df_val)+len(df_test):>5} rows")


Loading parquet from: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/mvsa_single_processed.parquet
Total rows loaded : 4511
Columns           : ['ID', 'text_sentiment', 'image_sentiment', 'final_sentiment', 'image_bytes', 'text_content']
Rows after dropna : 4511  (dropped 0)

Label distribution (final_sentiment):
  0  negative     1358  (30.1%)
  1  neutral       470  (10.4%)
  2  positive     2683  (59.5%)

Data split (stratified):
  Train :  3608 rows
  Val   :   451 rows
  Test  :   452 rows
  Total :  4511 rows


In [21]:
# ─────────────────────────────────────────────
# Cell 6 — Dataset Class & DataLoaders
# ─────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained(CONFIG["bert_model_name"])

# ImageNet normalization stats (for ResNet-101 pretrained on ImageNet)
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])


class MVSADataset(Dataset):
    """
    Loads pre-processed multimodal data from a pandas DataFrame (from parquet).
    image_bytes : raw PNG bytes stored in the parquet file
    text_content: raw text string
    label       : integer class (0=negative, 1=neutral, 2=positive)
    """

    def __init__(self, dataframe, tokenizer, image_transform, max_length):
        self.df        = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = image_transform
        self.max_len   = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Decode image from bytes ──────────────────────────
        img_bytes = row["image_bytes"]
        if isinstance(img_bytes, bytes):
            raw_bytes = img_bytes
        else:
            # numpy array of uint8
            raw_bytes = bytes(img_bytes)
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img_tensor = self.transform(img)

        # ── Tokenise text ────────────────────────────────────
        encoding = self.tokenizer(
            str(row["text_content"]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids"     : encoding["input_ids"].squeeze(0),      # [L]
            "attention_mask": encoding["attention_mask"].squeeze(0),  # [L]
            "token_type_ids": encoding["token_type_ids"].squeeze(0),  # [L]
            "image"         : img_tensor,                             # [3, H, W]
            "label"         : torch.tensor(row["label"], dtype=torch.long),
        }


# ── Build DataLoaders ────────────────────────────────────────────────
train_dataset = MVSADataset(df_train, tokenizer, train_transform, CONFIG["max_text_length"])
val_dataset   = MVSADataset(df_val,   tokenizer, eval_transform,  CONFIG["max_text_length"])
test_dataset  = MVSADataset(df_test,  tokenizer, eval_transform,  CONFIG["max_text_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Train batches : {len(train_loader)}  ({len(train_dataset)} samples)")
print(f"Val   batches : {len(val_loader)}  ({len(val_dataset)} samples)")
print(f"Test  batches : {len(test_loader)}  ({len(test_dataset)} samples)")

# ── Sanity check: inspect one batch ────────────────────────────────
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_ids      : {batch['input_ids'].shape}")
print(f"  attention_mask : {batch['attention_mask'].shape}")
print(f"  token_type_ids : {batch['token_type_ids'].shape}")
print(f"  image          : {batch['image'].shape}")
print(f"  label          : {batch['label'].shape}   values={batch['label'].tolist()}")


Train batches : 226  (3608 samples)
Val   batches : 29  (451 samples)
Test  batches : 29  (452 samples)

Batch shapes:
  input_ids      : torch.Size([16, 128])
  attention_mask : torch.Size([16, 128])
  token_type_ids : torch.Size([16, 128])
  image          : torch.Size([16, 3, 224, 224])
  label          : torch.Size([16])   values=[2, 2, 2, 2, 0, 2, 0, 0, 2, 1, 2, 2, 0, 1, 0, 0]


In [22]:
# ─────────────────────────────────────────────
# Cell 7 — Co-Attention Mechanism
# ─────────────────────────────────────────────
class CoAttentionLayer(nn.Module):
    """
    Bidirectional Cross-Modal Co-Attention Layer.

    Given text tokens T ∈ R^[B, Lt, d] and image tokens V ∈ R^[B, Lv, d]:
      • Text stream  : T attends to V  (text queries, image keys/values)
      • Image stream : V attends to T  (image queries, text keys/values)
    Both streams share the same dimension d and number of heads.
    Each stream is followed by:
        Add & LayerNorm → 2-layer FFN (GELU) → Add & LayerNorm
    """

    def __init__(self, hidden_dim, num_heads, ffn_dim, dropout):
        super().__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"

        # ── Text ← Image cross-attention ──────────────────────
        self.t2v_attn  = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.t2v_norm1 = nn.LayerNorm(hidden_dim)
        self.t2v_ffn   = nn.Sequential(
            nn.Linear(hidden_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, hidden_dim),
            nn.Dropout(dropout),
        )
        self.t2v_norm2 = nn.LayerNorm(hidden_dim)

        # ── Image ← Text cross-attention ──────────────────────
        self.v2t_attn  = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.v2t_norm1 = nn.LayerNorm(hidden_dim)
        self.v2t_ffn   = nn.Sequential(
            nn.Linear(hidden_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, hidden_dim),
            nn.Dropout(dropout),
        )
        self.v2t_norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, text_feats, image_feats, text_key_padding_mask=None):
        """
        Args:
            text_feats  : [B, Lt, d]
            image_feats : [B, Lv, d]
            text_key_padding_mask : [B, Lt] bool mask — True positions are IGNORED
                          (pass the ~attention_mask from BERT tokenizer)
        Returns:
            text_out    : [B, Lt, d]
            image_out   : [B, Lv, d]
        """
        # ─── Text stream: Q=text, K/V=image ───────────────────
        t_attn_out, _ = self.t2v_attn(
            query=text_feats,
            key=image_feats,
            value=image_feats,
        )
        text_out  = self.t2v_norm1(text_feats + t_attn_out)
        text_out  = self.t2v_norm2(text_out + self.t2v_ffn(text_out))

        # ─── Image stream: Q=image, K/V=text ──────────────────
        v_attn_out, _ = self.v2t_attn(
            query=image_feats,
            key=text_feats,
            value=text_feats,
            key_padding_mask=text_key_padding_mask,
        )
        image_out = self.v2t_norm1(image_feats + v_attn_out)
        image_out = self.v2t_norm2(image_out + self.v2t_ffn(image_out))

        return text_out, image_out


# ── Quick shape test ─────────────────────────────────────────────────
_B, _Lt, _Lv, _d = 2, 12, 49, 768
_layer = CoAttentionLayer(hidden_dim=_d, num_heads=8, ffn_dim=3072, dropout=0.1)
_t = torch.randn(_B, _Lt, _d)
_v = torch.randn(_B, _Lv, _d)
_to, _vo = _layer(_t, _v)
print(f"CoAttentionLayer OK  |  text_out: {tuple(_to.shape)}  image_out: {tuple(_vo.shape)}")
del _layer, _t, _v, _to, _vo


CoAttentionLayer OK  |  text_out: (2, 12, 768)  image_out: (2, 49, 768)


In [23]:
# ─────────────────────────────────────────────
# Cell 8 — ViLBERT Model Architecture
# ─────────────────────────────────────────────
class ViLBERT(nn.Module):
    """
    ViLBERT-style multimodal sentiment classifier.

    Architecture:
    ┌──────────────┐     ┌──────────────────────┐
    │  BERT-base   │     │  ResNet-101 (no FC)  │
    │  text stream │     │  image stream         │
    └──────┬───────┘     └──────────┬────────────┘
           │ [B, Lt, 768]           │ [B, HW, 2048]  → Linear → [B, HW, 768]
           │                        │
           └──────── Co-Attention ──┘   × N layers (bidirectional)
                          │
              CLS token [B,768] ++ mean-pool image [B,768]
                          │ concat  →  [B, 1536]
                          │
                     MLP Classifier
                          │
                     logits [B, 3]
    """

    def __init__(self, config):
        super().__init__()
        hd       = config["bert_hidden_size"]
        img_proj = config["image_proj_dim"]

        # ── Text encoder: BERT ──────────────────────────────────
        self.bert = BertModel.from_pretrained(config["bert_model_name"])

        # ── Image encoder: ResNet-101, strip final avgpool+fc ───
        resnet = tv_models.resnet101(weights=tv_models.ResNet101_Weights.DEFAULT)
        # Remove avgpool and fc — keep up to layer4 (spatial feature maps)
        self.visual_backbone = nn.Sequential(*list(resnet.children())[:-2])
        # Freeze BN stats in backbone to stabilise training
        for m in self.visual_backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()

        # ── Project image features from 2048 → hidden_dim ──────
        self.img_proj = nn.Sequential(
            nn.Linear(config["image_feature_dim"], img_proj),
            nn.LayerNorm(img_proj),
        )

        # ── Stacked Co-Attention layers ─────────────────────────
        self.co_attn_layers = nn.ModuleList([
            CoAttentionLayer(
                hidden_dim=hd,
                num_heads=config["co_attn_heads"],
                ffn_dim=config["ffn_hidden_dim"],
                dropout=config["co_attn_dropout"],
            )
            for _ in range(config["co_attn_layers"])
        ])

        # ── Fusion classifier head ───────────────────────────────
        fusion_in = config["fusion_input_dim"]   # 1536
        clf_hid   = config["classifier_hidden"]
        n_cls     = config["num_classes"]
        drop      = config["classifier_dropout"]

        self.classifier = nn.Sequential(
            nn.Linear(fusion_in, clf_hid),
            nn.GELU(),
            nn.Dropout(drop),
            nn.LayerNorm(clf_hid),
            nn.Linear(clf_hid, n_cls),
        )

    def forward(self, input_ids, attention_mask, token_type_ids, pixel_values):
        """
        Args:
            input_ids       : [B, L]
            attention_mask  : [B, L]   1=real token, 0=padding
            token_type_ids  : [B, L]
            pixel_values    : [B, 3, H, W]
        Returns:
            logits          : [B, num_classes]
        """
        B = input_ids.size(0)

        # ── Text stream ─────────────────────────────────────────
        bert_out   = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        text_feats = bert_out.last_hidden_state   # [B, L, 768]

        # ── Image stream ────────────────────────────────────────
        with torch.no_grad() if not self.training else torch.enable_grad():
            vis_map = self.visual_backbone(pixel_values)   # [B, 2048, H', W']
        H_p, W_p  = vis_map.shape[-2], vis_map.shape[-1]
        vis_flat  = vis_map.flatten(2).permute(0, 2, 1)   # [B, H'W', 2048]
        img_feats = self.img_proj(vis_flat)                # [B, H'W', 768]

        # ── Co-Attention: N bidirectional layers ────────────────
        # Build key_padding_mask for text (True = ignored position)
        txt_pad_mask = (attention_mask == 0)  # [B, L]

        for layer in self.co_attn_layers:
            text_feats, img_feats = layer(
                text_feats, img_feats,
                text_key_padding_mask=txt_pad_mask,
            )

        # ── Pooling & Fusion ────────────────────────────────────
        cls_token  = text_feats[:, 0, :]          # [B, 768]  — CLS token
        img_pool   = img_feats.mean(dim=1)         # [B, 768]  — mean-pooled visual
        fused      = torch.cat([cls_token, img_pool], dim=-1)  # [B, 1536]

        logits = self.classifier(fused)            # [B, 3]
        return logits


# ── Instantiate model ────────────────────────────────────────────────
device = torch.device(CONFIG["device"])
model  = ViLBERT(CONFIG).to(device)

# ── Parameter count ──────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ── Forward pass sanity check ────────────────────────────────────────
model.eval()
with torch.no_grad():
    _dummy = next(iter(train_loader))
    _logits = model(
        _dummy["input_ids"].to(device),
        _dummy["attention_mask"].to(device),
        _dummy["token_type_ids"].to(device),
        _dummy["image"].to(device),
    )
print(f"Forward pass OK  |  logits shape: {tuple(_logits.shape)}   (expected [B, 3])")
del _dummy, _logits
model.train()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to /root/.cache/torch/hub/checkpoints/resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:01<00:00, 174MB/s]  


Total parameters    : 182,698,563
Trainable parameters: 182,698,563
Forward pass OK  |  logits shape: (16, 3)   (expected [B, 3])


ViLBERT(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tr

In [24]:
# ─────────────────────────────────────────────
# Cell 9 — Training Loop
# ─────────────────────────────────────────────
# ── Optimizer: separate LR groups ────────────────────────────────────
bert_param_ids = set(id(p) for p in model.bert.parameters())
bert_params    = [p for p in model.parameters() if id(p) in bert_param_ids]
new_params     = [p for p in model.parameters() if id(p) not in bert_param_ids]

optimizer = AdamW([
    {"params": bert_params, "lr": CONFIG["bert_lr"]},
    {"params": new_params,  "lr": CONFIG["new_params_lr"]},
], weight_decay=CONFIG["weight_decay"])

# ── Scheduler: linear warmup then linear decay ───────────────────────
total_steps  = len(train_loader) * CONFIG["epochs"]
warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion = nn.CrossEntropyLoss()

# ── Helper: run one evaluation pass ─────────────────────────────────
def evaluate(loader, model, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids  = batch["input_ids"].to(device)
            attn_mask  = batch["attention_mask"].to(device)
            tok_ids    = batch["token_type_ids"].to(device)
            images     = batch["image"].to(device)
            labels     = batch["label"].to(device)

            logits = model(input_ids, attn_mask, tok_ids, images)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


# ── Training loop ───────────────────────────────────────────────────
best_val_f1      = 0.0
patience_counter = 0
best_ckpt_path   = os.path.join(CONFIG["checkpoint_dir"], "vilbert_best.pt")
history          = []

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}  {'Val F1':>7}  {'Status':>12}")
print("─" * 75)

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    # BN layers in visual_backbone stay in eval mode
    for m in model.visual_backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    train_loss, train_preds, train_labels = 0.0, [], []

    for batch in train_loader:
        input_ids  = batch["input_ids"].to(device)
        attn_mask  = batch["attention_mask"].to(device)
        tok_ids    = batch["token_type_ids"].to(device)
        images     = batch["image"].to(device)
        labels     = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn_mask, tok_ids, images)
        loss   = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        optimizer.step()
        scheduler.step()

        train_loss  += loss.item()
        train_preds.extend(logits.detach().argmax(dim=-1).cpu().tolist())
        train_labels.extend(labels.cpu().tolist())

    tr_loss = train_loss / len(train_loader)
    tr_acc  = accuracy_score(train_labels, train_preds)

    val_loss, val_acc, val_f1 = evaluate(val_loader, model, criterion, device)

    # ── Early stopping & checkpoint ─────────────────────────
    status = ""
    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        patience_counter = 0
        torch.save({
            "epoch"          : epoch,
            "model_state"    : model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_f1"         : val_f1,
            "val_acc"        : val_acc,
        }, best_ckpt_path)
        status = "✓ best saved"
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stop_patience"]:
            print(f"\nEarly stopping at epoch {epoch}  (no improvement for {CONFIG['early_stop_patience']} epochs)")
            break

    history.append({
        "epoch": epoch,
        "train_loss": tr_loss, "train_acc": tr_acc,
        "val_loss": val_loss,  "val_acc": val_acc, "val_f1": val_f1,
    })

    print(f"{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}  {val_f1:>7.4f}  {status:>12}")

print(f"\nTraining complete.  Best val macro-F1: {best_val_f1:.4f}")
print(f"Best checkpoint saved to: {best_ckpt_path}")


Epoch  Train Loss  Train Acc  Val Loss  Val Acc   Val F1        Status
───────────────────────────────────────────────────────────────────────────
    1      0.8220     0.6411    0.7171   0.6918   0.5308  ✓ best saved
    2      0.6537     0.7348    0.8494   0.6918   0.5046              
    3      0.5169     0.8065    0.8192   0.6674   0.5634  ✓ best saved
    4      0.3229     0.8900    1.5012   0.7095   0.5756  ✓ best saved
    5      0.2389     0.9224    1.2581   0.6851   0.5830  ✓ best saved
    6      0.2248     0.9299    1.5917   0.6984   0.5796              
    7      0.1611     0.9557    2.0646   0.6984   0.5902  ✓ best saved
    8      0.1344     0.9659    2.1154   0.7029   0.6027  ✓ best saved
    9      0.0934     0.9734    2.3912   0.6962   0.6011              
   10      0.0812     0.9773    2.2272   0.7095   0.6081  ✓ best saved
   11      0.0655     0.9845    2.5990   0.7095   0.6242  ✓ best saved
   12      0.0580     0.9848    2.7348   0.6829   0.5831              
 

In [25]:
# ─────────────────────────────────────────────
# Cell 10 — Evaluation on Test Set
# ─────────────────────────────────────────────

# ── Load best checkpoint ─────────────────────────────────────────────
print(f"Loading best checkpoint from: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(f"Checkpoint epoch : {ckpt['epoch']}   val_f1={ckpt['val_f1']:.4f}   val_acc={ckpt['val_acc']:.4f}")

# ── Predictions on test set ──────────────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attn_mask = batch["attention_mask"].to(device)
        tok_ids   = batch["token_type_ids"].to(device)
        images    = batch["image"].to(device)
        labels    = batch["label"].to(device)

        logits = model(input_ids, attn_mask, tok_ids, images)
        probs  = F.softmax(logits, dim=-1)
        preds  = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

# ── Metrics ──────────────────────────────────────────────────────────
class_names = [LABEL_IMAP[i] for i in range(CONFIG["num_classes"])]
test_acc    = accuracy_score(all_labels, all_preds)
test_f1     = f1_score(all_labels, all_preds, average="macro", zero_division=0)

SEP = "=" * 60
print(f"\n{SEP}")
print("  Test Set Results")
print(SEP)
print(f"  Accuracy  (overall)   : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Macro F1  (unweighted): {test_f1:.4f}")
print(f"\n  Per-Class Classification Report:")
print(classification_report(all_labels, all_preds,
                             target_names=class_names,
                             digits=4, zero_division=0))

# ── Confusion Matrix ─────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
print("  Confusion Matrix  (rows=true, cols=pred):")
header = "          " + "  ".join(f"{n:>10}" for n in class_names)
print(header)
for i, row_vals in enumerate(cm):
    row_str = "  ".join(f"{v:>10}" for v in row_vals)
    print(f"  {class_names[i]:>8}  {row_str}")

# ── Training history summary ─────────────────────────────────────────
if history:
    hist_df = pd.DataFrame(history)
    print(f"\n  Training History (last 5 epochs):")
    print(hist_df.tail(5).to_string(index=False))

print(f"\n{SEP}")
print(f"  Best checkpoint: {best_ckpt_path}")
print(SEP)


Loading best checkpoint from: /content/drive/MyDrive/Colab Notebooks/MVSA_Single/checkpoints/vilbert_best.pt
Checkpoint epoch : 11   val_f1=0.6242   val_acc=0.7095

  Test Set Results
  Accuracy  (overall)   : 0.7080  (70.80%)
  Macro F1  (unweighted): 0.6040

  Per-Class Classification Report:
              precision    recall  f1-score   support

    negative     0.6667    0.5882    0.6250       136
     neutral     0.3913    0.3830    0.3871        47
    positive     0.7762    0.8253    0.8000       269

    accuracy                         0.7080       452
   macro avg     0.6114    0.5988    0.6040       452
weighted avg     0.7032    0.7080    0.7044       452

  Confusion Matrix  (rows=true, cols=pred):
            negative     neutral    positive
  negative          80          11          45
   neutral          10          18          19
  positive          30          17         222

  Training History (last 5 epochs):
 epoch  train_loss  train_acc  val_loss  val_acc   val_f

In [26]:
# ─────────────────────────────────────────────
# Cell 11 — Save Model for Later Use
# ─────────────────────────────────────────────
import json

SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 1. Save full model checkpoint (weights + optimizer + metadata) ───
full_ckpt_path = os.path.join(SAVE_DIR, "vilbert_full.pt")
torch.save({
    "model_state_dict"    : model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config"              : CONFIG,
    "label_map"           : LABEL_MAP,
    "label_imap"          : LABEL_IMAP,
    "best_val_f1"         : best_val_f1,
    "test_acc"            : test_acc,
    "test_f1"             : test_f1,
    "history"             : history,
}, full_ckpt_path)
print(f"Full checkpoint saved : {full_ckpt_path}")

# ── 2. Save weights-only (lighter, for inference) ───────────────────
weights_path = os.path.join(SAVE_DIR, "vilbert_weights_only.pt")
torch.save(model.state_dict(), weights_path)
print(f"Weights-only saved    : {weights_path}")

# ── 3. Save CONFIG + label maps as JSON (for later reload) ──────────
config_to_save = {k: v for k, v in CONFIG.items() if not callable(v)}
config_to_save["label_map"]  = LABEL_MAP
config_to_save["label_imap"] = {str(k): v for k, v in LABEL_IMAP.items()}
config_path = os.path.join(SAVE_DIR, "config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config JSON saved     : {config_path}")

# ── 4. File sizes ────────────────────────────────────────────────────
for path in [full_ckpt_path, weights_path, config_path]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {os.path.basename(path):<35}  {size_mb:.1f} MB")

print("\n" + "=" * 60)
print("  HOW TO RELOAD THIS MODEL LATER")
print("=" * 60)
print("""
  import json, torch
  from transformers import BertModel, BertTokenizer

  # 1. Load config
  with open("config.json") as f:
      cfg = json.load(f)

  # 2. Rebuild model architecture
  model = ViLBERT(cfg)

  # 3a. Load full checkpoint (includes optimizer, history, metrics)
  ckpt = torch.load("vilbert_full.pt", map_location="cpu")
  model.load_state_dict(ckpt["model_state_dict"])
  label_map  = ckpt["label_map"]     # {"negative":0, "neutral":1, "positive":2}
  label_imap = ckpt["label_imap"]    # {0:"negative", 1:"neutral", 2:"positive"}

  # 3b. OR load weights only (for inference)
  model.load_state_dict(torch.load("vilbert_weights_only.pt", map_location="cpu"))

  # 4. Run inference
  model.eval()
  tokenizer = BertTokenizer.from_pretrained(cfg["bert_model_name"])
  # (prepare input_ids, attention_mask, token_type_ids, pixel_values)
  # logits = model(input_ids, attention_mask, token_type_ids, pixel_values)
  # pred   = logits.argmax(dim=-1).item()
  # label  = label_imap[pred]
""")


Full checkpoint saved : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final/vilbert_full.pt
Weights-only saved    : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final/vilbert_weights_only.pt
Config JSON saved     : /content/drive/MyDrive/Colab Notebooks/MVSA_Single/vilbert_final/config.json
  vilbert_full.pt                      2087.5 MB
  vilbert_weights_only.pt              697.7 MB
  config.json                          0.0 MB

  HOW TO RELOAD THIS MODEL LATER

  import json, torch
  from transformers import BertModel, BertTokenizer

  # 1. Load config
  with open("config.json") as f:
      cfg = json.load(f)

  # 2. Rebuild model architecture
  model = ViLBERT(cfg)

  # 3a. Load full checkpoint (includes optimizer, history, metrics)
  ckpt = torch.load("vilbert_full.pt", map_location="cpu")
  model.load_state_dict(ckpt["model_state_dict"])
  label_map  = ckpt["label_map"]     # {"negative":0, "neutral":1, "positive":2}
  label_imap = ckpt["label_imap"]  